# Water Quality Prediction: Benchmark Notebook

## 1. Setup

In [ ]:
!pip install uv rioxarray shap

In [ ]:

import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from IPython.display import display

import xarray as xr
import rioxarray as rxr
import rasterio
from rasterio.windows import Window

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
from scipy.spatial import cKDTree

from datetime import date
from tqdm import tqdm
import os
import shap
from xgboost import XGBRegressor

from google.colab import drive
drive.mount('/content/drive')

## 2. Load Data

In [19]:
BASE_DIR = '/content/drive/MyDrive/EY-AI-and-Data-Challenge-main'

Water_Quality_df = pd.read_csv(f'{BASE_DIR}/water_quality_training_dataset.csv')
display(Water_Quality_df.head(5))


,Latitude,Longitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus
0,-28.760833,17.730278,02-01-2011,128.912,555.0,10.0
1,-26.861111,28.884722,03-01-2011,74.720,162.9,163.0
2,-26.450000,28.085833,03-01-2011,89.254,573.0,80.0
3,-27.671111,27.236944,03-01-2011,82.000,203.6,101.0
4,-27.356667,27.286389,03-01-2011,56.100,145.1,151.0


In [20]:
landsat_train_features = pd.read_csv(f'{BASE_DIR}/LANDSAT/landsat_features_training_200m.csv')
display(landsat_train_features.head(5))


,Latitude,Longitude,Sample Date,med_blue,med_green,med_red,med_nir,med_swir16,med_swir22,med_st_b10,...,med_EVI,median_LST,urban_ratio,water_ratio,veg_ratio,bare_ratio,hot_ratio,corr_NDVI_LST,corr_NDBI_LST,corr_MNDWI_LST
0,-28.760833,17.730278,02-01-2011,9557.0,11426.0,12802.0,11190.0,7836.0,7786.0,44447.5,...,-0.246853,44447.5,0.000000,0.767857,0.053571,0.0,0.250000,0.819405,0.527119,-0.915259
1,-26.861111,28.884722,03-01-2011,8569.0,9482.0,9083.5,18750.0,13831.0,10617.5,45270.0,...,2.689771,45270.0,0.000000,0.000000,0.839286,0.0,0.250000,0.036884,0.515972,-0.381128
2,-26.450000,28.085833,03-01-2011,9502.5,10610.0,12540.0,15210.0,17974.0,14067.0,41446.0,...,0.347978,41446.0,1.000000,0.000000,0.000000,0.0,0.232143,-0.119653,-0.146525,0.165935
3,-27.671111,27.236944,03-01-2011,9430.5,10669.0,10794.0,15423.5,13568.5,10619.0,45752.5,...,1.223473,45752.5,0.142857,0.196429,0.053571,0.0,0.232143,-0.314433,0.677673,-0.164215
4,-27.356667,27.286389,03-01-2011,8815.5,9778.5,9404.5,16688.5,13497.5,10104.0,44581.5,...,2.601336,44581.5,0.000000,0.000000,0.214286,0.0,0.250000,-0.515442,0.761927,-0.477777


In [21]:
Terraclimate_df = pd.read_csv(f'{BASE_DIR}/TerraClimate/terraclimate_features_training_full.csv')
display(Terraclimate_df.head(5))


,Latitude,Longitude,Sample Date,aet,def,pdsi,pet,ppt,q,soil,srad,swe,tmax,tmin,vap,vpd,ws
0,-28.760833,17.730278,02-01-2011,31.100000,143.100000,3.65,174.2,32.7,1.6,0.0,270.20070,0.0,36.100000,22.689999,1.471,2.92,2.02
1,-26.861111,28.884722,03-01-2011,53.800000,70.300000,0.66,124.1,51.1,2.6,12.8,227.50043,0.0,27.160000,13.219999,1.620,0.95,1.85
2,-26.450000,28.085833,03-01-2011,60.800000,66.700005,-1.16,127.5,62.7,3.1,6.8,230.89938,0.0,27.519999,14.090000,1.632,1.02,1.82
3,-27.671111,27.236944,03-01-2011,83.200005,46.500000,2.84,129.7,84.2,4.2,7.2,220.29940,0.0,28.869999,14.639999,1.614,1.22,1.89
4,-27.356667,27.286389,03-01-2011,77.300000,51.900000,2.65,129.2,78.0,3.9,7.8,222.90000,0.0,28.670000,14.690000,1.636,1.18,1.84


In [22]:
target_cols = ["Total Alkalinity", "Electrical Conductance", "Dissolved Reactive Phosphorus"]

# ── Load water quality monthly averages ──────────────────────────────────────
wq_monthly = pd.read_excel(
    f'{BASE_DIR}/water_quality_monthly_averages.xlsx',
    sheet_name='Monthly_Averages'
)
display(wq_monthly.head(3))

# Columns to use as features (exclude metadata / id columns)
WQ_META_COLS = ['STATION_ID', 'POINT_ID', 'SOURCE', 'WATER_BODY', 'LOCATION',
                'LAT', 'LON', 'YEAR', 'MONTH']
WQ_FEAT_COLS = [c for c in wq_monthly.columns if c not in WQ_META_COLS]
print(f"Water-quality feature columns ({len(WQ_FEAT_COLS)}): {WQ_FEAT_COLS}")

# ── Nearest-station feature matcher ──────────────────────────────────────────
def match_wq_features(samples_df, wq_monthly, date_col='Sample Date',
                      lat_col='Latitude', lon_col='Longitude',
                      feat_cols=WQ_FEAT_COLS, prefix='wq_'):
    """
    For every row in samples_df, find the geographically nearest station in
    wq_monthly (by decimal lat/lon), then look up that station's monthly-
    average measurements for the same YEAR and MONTH as the sample.
    Falls back to the station's overall mean when no month match exists.
    Returns a DataFrame with len(samples_df) rows and len(feat_cols) columns.
    """
    # Unique stations with valid GPS
    stations = (wq_monthly[['STATION_ID', 'LAT', 'LON']]
                .dropna(subset=['LAT', 'LON'])
                .drop_duplicates('STATION_ID')
                .reset_index(drop=True))

    tree = cKDTree(stations[['LAT', 'LON']].values)

    sample_coords = samples_df[[lat_col, lon_col]].values
    _, idx = tree.query(sample_coords)
    nearest_ids = stations.iloc[idx]['STATION_ID'].values

    dates  = pd.to_datetime(samples_df[date_col], dayfirst=True)
    years  = dates.dt.year.values
    months = dates.dt.month.values

    # Build (STATION_ID, YEAR, MONTH) → row lookup
    key_df = pd.DataFrame({
        'STATION_ID': nearest_ids,
        'YEAR':       years,
        'MONTH':      months,
    })

    merged = key_df.merge(
        wq_monthly[['STATION_ID', 'YEAR', 'MONTH'] + feat_cols],
        on=['STATION_ID', 'YEAR', 'MONTH'],
        how='left'
    )

    # Fill remaining NaN with per-station overall mean
    station_means = wq_monthly.groupby('STATION_ID')[feat_cols].mean()
    missing_mask  = merged[feat_cols].isna().any(axis=1)
    if missing_mask.any():
        fill_vals = (merged.loc[missing_mask, 'STATION_ID']
                          .map(station_means.to_dict(orient='index'))
                          .apply(pd.Series))
        fill_vals.index = merged.index[missing_mask]
        merged.loc[missing_mask, feat_cols] = fill_vals[feat_cols]

    result = merged[feat_cols].copy()
    result.columns = [prefix + c for c in feat_cols]
    result = result.reset_index(drop=True)
    return result

# Match training samples
print("Matching training samples to nearest water-quality station …")
wq_train_features = match_wq_features(Water_Quality_df, wq_monthly)
print(f"  wq_train_features: {wq_train_features.shape}")
display(wq_train_features.head(5))

,STATION_ID,POINT_ID,SOURCE,WATER_BODY,LOCATION,LAT,LON,YEAR,MONTH,2[SO4],...,[NH4],[NO3],[PO4],[SI],[SO4],[TAL],adj. SAR,col_10,dEC,dTDS
0,A2H006Q01,90160,RIVERS,RIVER,PIENAARS RIVER AT KLIPDRIFT,25.3806,28.3167,1999,1,16.959710,...,0.001111,0.001302,0.002558,0.178915,0.366285,1.3455,1.438208,37.150,-8.816422,-6.157379
1,A2H006Q01,90160,RIVERS,RIVER,PIENAARS RIVER AT KLIPDRIFT,25.3806,28.3167,1999,2,14.480999,...,0.001597,0.001105,0.001303,0.188434,0.364984,1.6330,1.513747,41.275,-8.027078,-6.689824
2,A2H006Q01,90160,RIVERS,RIVER,PIENAARS RIVER AT KLIPDRIFT,25.3806,28.3167,1999,3,11.814228,...,0.001511,0.001500,0.000842,0.154520,0.307388,1.7496,1.445738,40.500,-8.766975,-7.634444


Water-quality feature columns (44): ['2[SO4]', 'CA', 'CL', 'Charge Balance', 'EC', 'EC (calc.)', 'EC_mS_cm', 'F', 'I', 'K', 'MG', 'NH4', 'NO3+NO2', 'PH', 'PO4', 'SAR', 'SI', 'SO4', 'Sz+', 'Sz-', 'TAL', 'TDS', 'TDS (calc.)', 'TDS/EC', '[CA]', '[CL]', '[CL]_1', '[CO3]', '[F]', '[HCO3]', '[HCO3]+2[CO3]', '[K]', '[MG]', '[NA]', '[NH4]', '[NO3]', '[PO4]', '[SI]', '[SO4]', '[TAL]', 'adj. SAR', 'col_10', 'dEC', 'dTDS']
Matching training samples to nearest water-quality station …
  wq_train_features: (9319, 44)


,wq_2[SO4],wq_CA,wq_CL,wq_Charge Balance,wq_EC,wq_EC (calc.),wq_EC_mS_cm,wq_F,wq_I,wq_K,...,wq_[NH4],wq_[NO3],wq_[PO4],wq_[SI],wq_[SO4],wq_[TAL],wq_adj. SAR,wq_col_10,wq_dEC,wq_dTDS
0,12.749032,33.195041,125.466,-0.410442,79.907973,736.44038,799.07973,0.344028,0.010277,5.229473,...,0.005306,0.003932,0.000441,0.16223,0.495228,1.628092,2.614887,85.728973,-8.078886,-4.767762
1,12.749032,33.195041,125.466,-0.410442,79.907973,736.44038,799.07973,0.344028,0.010277,5.229473,...,0.005306,0.003932,0.000441,0.16223,0.495228,1.628092,2.614887,85.728973,-8.078886,-4.767762
2,12.749032,33.195041,125.466,-0.410442,79.907973,736.44038,799.07973,0.344028,0.010277,5.229473,...,0.005306,0.003932,0.000441,0.16223,0.495228,1.628092,2.614887,85.728973,-8.078886,-4.767762
3,12.749032,33.195041,125.466,-0.410442,79.907973,736.44038,799.07973,0.344028,0.010277,5.229473,...,0.005306,0.003932,0.000441,0.16223,0.495228,1.628092,2.614887,85.728973,-8.078886,-4.767762
4,12.749032,33.195041,125.466,-0.410442,79.907973,736.44038,799.07973,0.344028,0.010277,5.229473,...,0.005306,0.003932,0.000441,0.16223,0.495228,1.628092,2.614887,85.728973,-8.078886,-4.767762


## 3. Feature Engineering

In [23]:
def combine_three_datasets(*datasets):
    data = pd.concat(list(datasets), axis=1)
    data = data.loc[:, ~data.columns.duplicated()]
    return data

In [24]:
def classify_region(lat, lon):
    """南非主要地理/氣候區域分類"""
    try:
        lat, lon = float(lat), float(lon)
    except:
        return "Unknown"

    if lon < 20.5 and lat <= -32.0:
        return "West_Coast"
    elif 20.5 <= lon <= 27.5 and -34.8 <= lat <= -32.0:
        return "South_Coast"
    elif lon >= 29.0 and -31.0 <= lat <= -26.5:
        return "East_Coast"
    elif 24.0 <= lon < 29.0 and -34.0 <= lat < -30.5:
        return "Eastern_Cape"
    elif lat > -29.0:
        return "Interior"
    else:
        return "Northern_Arid"

def get_season(month):
    """南半球季節"""
    if month in [12, 1, 2]:   return "Summer"
    elif month in [3, 4, 5]:  return "Autumn"
    elif month in [6, 7, 8]:  return "Winter"
    else:                     return "Spring"

In [25]:
wq_data = combine_three_datasets(Water_Quality_df, landsat_train_features,  wq_train_features)
display(wq_data.head(5))

,Latitude,Longitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus,med_blue,med_green,med_red,med_nir,...,wq_[NH4],wq_[NO3],wq_[PO4],wq_[SI],wq_[SO4],wq_[TAL],wq_adj. SAR,wq_col_10,wq_dEC,wq_dTDS
0,-28.760833,17.730278,02-01-2011,128.912,555.0,10.0,9557.0,11426.0,12802.0,11190.0,...,0.005306,0.003932,0.000441,0.16223,0.495228,1.628092,2.614887,85.728973,-8.078886,-4.767762
1,-26.861111,28.884722,03-01-2011,74.720,162.9,163.0,8569.0,9482.0,9083.5,18750.0,...,0.005306,0.003932,0.000441,0.16223,0.495228,1.628092,2.614887,85.728973,-8.078886,-4.767762
2,-26.450000,28.085833,03-01-2011,89.254,573.0,80.0,9502.5,10610.0,12540.0,15210.0,...,0.005306,0.003932,0.000441,0.16223,0.495228,1.628092,2.614887,85.728973,-8.078886,-4.767762
3,-27.671111,27.236944,03-01-2011,82.000,203.6,101.0,9430.5,10669.0,10794.0,15423.5,...,0.005306,0.003932,0.000441,0.16223,0.495228,1.628092,2.614887,85.728973,-8.078886,-4.767762
4,-27.356667,27.286389,03-01-2011,56.100,145.1,151.0,8815.5,9778.5,9404.5,16688.5,...,0.005306,0.003932,0.000441,0.16223,0.495228,1.628092,2.614887,85.728973,-8.078886,-4.767762


In [26]:

# Extract temporal features
_dates = pd.to_datetime(wq_data['Sample Date'], dayfirst=True)
wq_data['month']  = _dates.dt.month
wq_data['year']   = _dates.dt.year
wq_data['doy']    = _dates.dt.dayofyear

# Cyclical encodings
wq_data['month_sin'] = np.sin(2 * np.pi * wq_data['month'] / 12)
wq_data['month_cos'] = np.cos(2 * np.pi * wq_data['month'] / 12)
wq_data['doy_sin']   = np.sin(2 * np.pi * wq_data['doy'] / 365)
wq_data['doy_cos']   = np.cos(2 * np.pi * wq_data['doy'] / 365)

wq_data['season'] = wq_data['month'].apply(get_season)

# Compute region — kept in df for splitting ONLY, not used as a model feature
wq_data['region'] = wq_data.apply(
    lambda row: classify_region(row['Latitude'], row['Longitude']), axis=1)

# One-hot encode season only (no region or region×season — avoids geographic leakage)
wq_data = pd.get_dummies(wq_data, columns=['season'], prefix='season')

print("region distribution:")
print(wq_data['region'].value_counts())
print(f"\nTotal samples: {len(wq_data)}")
print(f"Columns: {wq_data.shape[1]}")


region distribution:
region
Interior         5048
East_Coast       1765
Northern_Arid     975
West_Coast        783
South_Coast       447
Eastern_Cape      301
Name: count, dtype: int64

Total samples: 9319
Columns: 85


In [27]:
wq_data = wq_data.fillna(wq_data.median(numeric_only=True))
print(wq_data.isna().sum().sum(), "missing values remaining")

0 missing values remaining


## 4. Model Training

In [28]:

from itertools import product as iterproduct

# ── Region-based train / test split ──────────────────────────────────────────
TRAIN_REGIONS = {"East_Coast", "Northern_Arid", "Interior", "South_Coast", "West_Coast"}
TEST_REGIONS  = {"Eastern_Cape"}

train_mask = wq_data['region'].isin(TRAIN_REGIONS)
test_mask  = wq_data['region'].isin(TEST_REGIONS)

# Features: drop metadata cols + region + targets  (region = spatial leakage)
_target_cols = ['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']
_meta_cols   = ['Latitude', 'Longitude', 'Sample Date', 'region']
_drop        = [c for c in _meta_cols + _target_cols if c in wq_data.columns]

X_all          = wq_data.drop(columns=_drop)
train_features = X_all.columns.tolist()

X_train = X_all[train_mask].reset_index(drop=True)
X_test  = X_all[test_mask].reset_index(drop=True)

y_TA_train  = wq_data.loc[train_mask, 'Total Alkalinity'].reset_index(drop=True)
y_TA_test   = wq_data.loc[test_mask,  'Total Alkalinity'].reset_index(drop=True)
y_EC_train  = wq_data.loc[train_mask, 'Electrical Conductance'].reset_index(drop=True)
y_EC_test   = wq_data.loc[test_mask,  'Electrical Conductance'].reset_index(drop=True)
y_DRP_train = wq_data.loc[train_mask, 'Dissolved Reactive Phosphorus'].reset_index(drop=True)
y_DRP_test  = wq_data.loc[test_mask,  'Dissolved Reactive Phosphorus'].reset_index(drop=True)

print(f"Train: {X_train.shape[0]:4d} samples | Regions: {sorted(wq_data.loc[train_mask,'region'].unique())}")
print(f"Test:  {X_test.shape[0]:4d} samples | Regions: {sorted(wq_data.loc[test_mask, 'region'].unique())}")
print(f"Feature count: {len(train_features)}")

# ── Grid Search (~64 combinations per target) ─────────────────────────────────
PARAM_GRID = {
    'n_estimators':     [200, 300],        # base=300, spread ±100
    'max_depth':        [10, 15, 20, None],     # base=15, spread ±5 + unbounded
    'min_samples_leaf': [4, 6, 8, 10],          # base=6, spread below and above
    'max_features':     ['sqrt'],       # base=sqrt
}
_gs_keys   = list(PARAM_GRID.keys())
_gs_combos = list(iterproduct(*[PARAM_GRID[k] for k in _gs_keys]))
print(f"\nGrid search: {len(_gs_combos)} hyperparameter combinations")


def grid_search_rf(X_tr, y_tr, X_te, y_te, param_name="Parameter"):
    """Per-target RF grid search; best model chosen by test R²."""
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_te_s = scaler.transform(X_te)

    records      = []
    best_r2_test = -np.inf
    best_model   = None
    best_params  = None

    print(f"\n{'='*75}")
    print(f"  Grid Search — {param_name}  ({len(_gs_combos)} combos)")
    print(f"{'='*75}")
    print(f"  {'#':>3} | n_est | depth | leaf | feat  | Train R²  | Test R² ")
    print(f"  {'-'*65}")

    for i, combo in enumerate(_gs_combos, 1):
        params = dict(zip(_gs_keys, combo))
        rf = RandomForestRegressor(**params, n_jobs=-1, random_state=42)
        rf.fit(X_tr_s, y_tr)

        y_tr_pred = rf.predict(X_tr_s)
        y_te_pred = rf.predict(X_te_s)
        r2_tr   = r2_score(y_tr, y_tr_pred)
        r2_te   = r2_score(y_te, y_te_pred)
        rmse_tr = np.sqrt(mean_squared_error(y_tr, y_tr_pred))
        rmse_te = np.sqrt(mean_squared_error(y_te, y_te_pred))

        records.append({
            **params,
            'R2_Train': round(r2_tr, 4), 'RMSE_Train': round(rmse_tr, 4),
            'R2_Test':  round(r2_te, 4), 'RMSE_Test':  round(rmse_te, 4),
        })

        marker = "  ◀ BEST" if r2_te > best_r2_test else ""
        print(f"  [{i:3d}] | {params['n_estimators']:5d} | "
              f"{str(params['max_depth']):5s} | {params['min_samples_leaf']:4d} | "
              f"{params['max_features']:5s} |   {r2_tr:6.3f}  |  {r2_te:6.3f}{marker}")

        if r2_te > best_r2_test:
            best_r2_test = r2_te
            best_model   = rf
            best_params  = dict(params)

    results_df = (pd.DataFrame(records)
                    .sort_values('R2_Test', ascending=False)
                    .reset_index(drop=True))

    print(f"\n  ✔ Best params : {best_params}")
    print(f"  ✔ Best Test R² = {best_r2_test:.4f}")
    return best_model, scaler, best_params, results_df


Train: 9018 samples | Regions: ['East_Coast', 'Interior', 'Northern_Arid', 'South_Coast', 'West_Coast']
Test:   301 samples | Regions: ['Eastern_Cape']
Feature count: 78

Grid search: 32 hyperparameter combinations


In [29]:
X_all.dtypes

,0
med_blue,float64
med_green,float64
med_red,float64
med_nir,float64
med_swir16,float64
...,...
doy_cos,float64
season_Autumn,bool
season_Spring,bool
season_Summer,bool


In [ ]:

def plot_shap_importance(model, scaler, X, feature_names, target_name, max_display=25):
    """SHAP-based feature importance using TreeExplainer (bar + beeswarm)."""
    X_scaled = scaler.transform(X)
    X_df = pd.DataFrame(X_scaled, columns=feature_names)

    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_df)

    # Bar plot — mean |SHAP| per feature
    plt.figure()
    shap.summary_plot(shap_values, X_df, plot_type='bar',
                      max_display=max_display, show=False)
    plt.title(f"SHAP Feature Importance (Mean |SHAP|) — {target_name}")
    plt.tight_layout()
    plt.show()

    # Beeswarm plot — direction + magnitude
    plt.figure()
    shap.summary_plot(shap_values, X_df, max_display=max_display, show=False)
    plt.title(f"SHAP Beeswarm — {target_name}")
    plt.tight_layout()
    plt.show()


In [ ]:

# Run independent grid search for each target (~64 combos × 3 targets = 192 RF fits)
model_TA,  scaler_TA,  best_TA,  gs_TA  = grid_search_rf(
    X_train, y_TA_train,  X_test, y_TA_test,  "Total Alkalinity")

model_EC,  scaler_EC,  best_EC,  gs_EC  = grid_search_rf(
    X_train, y_EC_train,  X_test, y_EC_test,  "Electrical Conductance")

model_DRP, scaler_DRP, best_DRP, gs_DRP = grid_search_rf(
    X_train, y_DRP_train, X_test, y_DRP_test, "Dissolved Reactive Phosphorus")


In [ ]:

# SHAP feature importance — bar (mean |SHAP|) + beeswarm for each target
# Uses the test set so SHAP values reflect out-of-sample behaviour
plot_shap_importance(model_TA,  scaler_TA,  X_test, train_features, "Total Alkalinity")
plot_shap_importance(model_EC,  scaler_EC,  X_test, train_features, "Electrical Conductance")
plot_shap_importance(model_DRP, scaler_DRP, X_test, train_features, "Dissolved Reactive Phosphorus")


In [ ]:

# ── All trials (sorted by Test R²) ───────────────────────────────────────────
print("=" * 70)
print("All Grid Search Trials — Total Alkalinity  (sorted by Test R²)")
display(gs_TA)

print("=" * 70)
print("All Grid Search Trials — Electrical Conductance  (sorted by Test R²)")
display(gs_EC)

print("=" * 70)
print("All Grid Search Trials — Dissolved Reactive Phosphorus  (sorted by Test R²)")
display(gs_DRP)

# ── Best hyperparameters summary ──────────────────────────────────────────────
summary = pd.DataFrame([
    {"Target": "Total Alkalinity",
     **best_TA,
     "Train R²": gs_TA.iloc[0]['R2_Train'],
     "Test R²":  gs_TA.iloc[0]['R2_Test']},
    {"Target": "Electrical Conductance",
     **best_EC,
     "Train R²": gs_EC.iloc[0]['R2_Train'],
     "Test R²":  gs_EC.iloc[0]['R2_Test']},
    {"Target": "Dissolved Reactive Phosphorus",
     **best_DRP,
     "Train R²": gs_DRP.iloc[0]['R2_Train'],
     "Test R²":  gs_DRP.iloc[0]['R2_Test']},
])
print("\n" + "=" * 70)
print("Best Hyperparameters per Target")
display(summary)


## 5. Submission

In [ ]:
test_file            = pd.read_csv(f'{BASE_DIR}/submission_template.csv')
landsat_val_features = pd.read_csv(f'{BASE_DIR}/LANDSAT/landsat_features_validation_200m.csv')
Terraclimate_val_df  = pd.read_csv(f'{BASE_DIR}/TerraClimate/terraclimate_features_validation_full.csv')

# Match validation samples to nearest water-quality station
print("Matching validation samples to nearest water-quality station …")
wq_val_features = match_wq_features(test_file, wq_monthly)
print(f"  wq_val_features: {wq_val_features.shape}")

val_data = combine_three_datasets(test_file, landsat_val_features, Terraclimate_val_df, wq_val_features)
val_data = val_data.fillna(val_data.median(numeric_only=True))
display(val_data.head(5))

In [ ]:

val_data = val_data.drop(columns=target_cols, errors='ignore')

# Temporal features — must match training pipeline exactly
_val_dates = pd.to_datetime(val_data['Sample Date'], dayfirst=True)
val_data['month']     = _val_dates.dt.month
val_data['year']      = _val_dates.dt.year
val_data['doy']       = _val_dates.dt.dayofyear
val_data['month_sin'] = np.sin(2 * np.pi * val_data['month'] / 12)
val_data['month_cos'] = np.cos(2 * np.pi * val_data['month'] / 12)
val_data['doy_sin']   = np.sin(2 * np.pi * val_data['doy'] / 365)
val_data['doy_cos']   = np.cos(2 * np.pi * val_data['doy'] / 365)

val_data['season'] = val_data['month'].apply(get_season)
val_data = pd.get_dummies(val_data, columns=['season'], prefix='season')

# Build feature matrix aligned to training columns
_val_meta_cols = ['Latitude', 'Longitude', 'Sample Date']
_val_drop = [c for c in _val_meta_cols + target_cols if c in val_data.columns]
submission_val_data = val_data.drop(columns=_val_drop, errors='ignore')

# Align to exact training feature set (fills any missing season dummies with 0)
submission_val_data = submission_val_data.reindex(columns=train_features, fill_value=0)

print(f"Submission features: {submission_val_data.shape[1]}  (train features: {len(train_features)})")
display(submission_val_data.head())


In [ ]:
pred_TA_submission  = model_TA.predict(scaler_TA.transform(submission_val_data))
pred_EC_submission  = model_EC.predict(scaler_EC.transform(submission_val_data))
pred_DRP_submission = model_DRP.predict(scaler_DRP.transform(submission_val_data))

In [ ]:
submission_df = pd.DataFrame({
    'Latitude':                     test_file['Latitude'].values,
    'Longitude':                    test_file['Longitude'].values,
    'Sample Date':                  test_file['Sample Date'].values,
    'Total Alkalinity':             pred_TA_submission,
    'Electrical Conductance':       pred_EC_submission,
    'Dissolved Reactive Phosphorus': pred_DRP_submission
})
display(submission_df.head())

In [ ]:
submission_df.to_csv("submission_v30.csv", index=False)
